# Utilities

> Utilities for processing Euclid images.

In [ ]:
# | default_exp euclid.utilities

In [ ]:
# | export

import os

import pandas as pd

from astropy.io import fits
from glob import glob

In [ ]:
# | export


def get_nisp_images_for_observation(
    obs_id,  # the main observation id
    n_prior=0,  # number of previous observations to include
    n_after=0,  # number of subsequent observations to include
    path="~/data/euclid",  # base path to search
    recursive=False,  # search recursively
):
    """Find NISP images for the specified obs_id and optionally n_prior and n_after observations."""
    info = dict(filename=[], filter=[], dithobs=[], obs_id=[], mjd=[])
    obs_id = int(obs_id)
    for i in range(obs_id - n_prior, obs_id + n_after + 1):
        if recursive:
            fns = glob(
                os.path.join(path, "**", f"EUC_NIR*-{i}-*Z.fits"), recursive=True
            )
        else:
            fns = glob(os.path.join(path, f"EUC_NIR*-{i}-*Z.fits"))
        if len(fns) == 0:
            print(f"Found no files for observation id {i}.")
        if len(fns) < 12:
            print(f"Missing some files for observation id {i}.")
        elif len(fns) > 12:
            print(f"Found too many files for observation id {i}.")
        for fn in fns:
            with fits.open(fn) as f:
                h = f[0].header
                info["filename"].append(fn)
                info["filter"].append(h["FILTER"])
                info["dithobs"].append(h["DITHOBS"])
                info["obs_id"].append(h["OBS_ID"])
                info["mjd"].append(h["MJD-OBS"])
    info = pd.DataFrame(info)
    info = info.sort_values("mjd").reset_index(drop=True)
    return info

In [ ]:
# | export


def get_primary_header(
    fns,  # an iterable of image filenames
):
    """Create a primary header from the provided list of files.

    The returned header contains all the entries from the files' primary headers
    that have consistent values across all the files.
    """
    hdr = fits.Header()
    mismatch = set()
    for fn in fns:
        newhdr = fits.getheader(fn)
        for h in newhdr:
            if h not in hdr:
                try:
                    hdr[h] = newhdr[h]
                except ValueError:
                    pass
            elif hdr[h] != newhdr[h]:
                mismatch.add(h)
    for h in mismatch:
        hdr.remove(h)
    return hdr

In [ ]:
# | export


def get_dq_mask(fn, extname, maskbits):
    extname = extname.replace("SCI", "DQ")
    dq = fits.getdata(fn, extname=extname)
    maskbits = np.atleast_1d(maskbits)
    mask = dq & 2 ** maskbits[0] > 0
    for b in maskbits[1:]:
        mask |= dq & 2**b > 0
    return mask


def get_persistence_mask(fn, extname):
    return get_dq_mask(fn, extname, [13])


def get_invalid_mask(fn, extname):
    return get_dq_mask(fn, extname, [2, 3, 4, 6, 7, 8, 9, 10, 16])


def get_rms(fn, extname):
    extname = extname.replace("SCI", "RMS")
    rms = fits.getdata(fn, extname=extname)
    return rms


def fits_append(fn, data, ext, primary_header, exthdr=None):
    if exthdr is None:
        exthdr = fits.Header([("EXTNAME", ext)])
    else:
        exthdr.update(EXTNAME=ext)
    if not os.path.exists(fn):
        fits.append(fn, None, primary_header)
    fits.append(fn, data, exthdr)

In [ ]:
# | export


def remove_if_necessary(path, glob):
    fns = glob.glob(os.path.join(path, glob))
    for fn in fns:
        os.remove(fn)

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()